## Load dataset 

In [0]:
df=spark.read.csv('s3://supply-chain-management-project/',header=True, inferSchema=True)
df.show()

## load tables

In [0]:
%fs ls s3://supply-chain-management-project

In [0]:
trucks_df= spark.read.format('csv').option("header", "true").load("s3://supply-chain-management-project/trucks.csv")
trucks_df.createOrReplaceTempView("trucks")
display(trucks_df)

In [0]:
truck_utilization_metrics_df= spark.read.format('csv').option("header", "true").load("s3://supply-chain-management-project/truck_utilization_metrics.csv")
truck_utilization_metrics_df.createOrReplaceTempView("truck_utilization_metrics")
display(truck_utilization_metrics_df)

In [0]:
trips_df= spark.read.format('csv').option("header", "true").load("s3://supply-chain-management-project/trips.csv")
trips_df.createOrReplaceTempView("trips")
display(trips_df)

In [0]:
trailers_df= spark.read.format('csv').option("header", "true").load("s3://supply-chain-management-project/trailers.csv")
trailers_df.createOrReplaceTempView("trailers")
display(trailers_df)

In [0]:
safety_incidents_df= spark.read.format('csv').option("header", "true").load("s3://supply-chain-management-project/safety_incidents.csv")
safety_incidents_df.createOrReplaceTempView("safety_incidents")
display(safety_incidents_df)

In [0]:
routes_df= spark.read.format('csv').option("header", "true").load("s3://supply-chain-management-project/routes.csv")
routes_df.createOrReplaceTempView("routes")
display(routes_df)

In [0]:
maintenance_records_df= spark.read.format('csv').option("header", "true").load("s3://supply-chain-management-project/maintenance_records.csv")
maintenance_records_df.createOrReplaceTempView("maintenance_records")
display(maintenance_records_df)

In [0]:
loads_df= spark.read.format('csv').option("header", "true").load("s3://supply-chain-management-project/loads.csv")
loads_df.createOrReplaceTempView("loads")
display(loads_df)

In [0]:
fuel_purchases_df= spark.read.format('csv').option("header", "true").load("s3://supply-chain-management-project/fuel_purchases.csv")
fuel_purchases_df.createOrReplaceTempView("fuel_purchases")
display(fuel_purchases_df)

In [0]:
facilities_df= spark.read.format('csv').option("header", "true").load("s3://supply-chain-management-project/facilities.csv")
facilities_df.createOrReplaceTempView("facilities")
display(facilities_df)

In [0]:
delivery_Event_df= spark.read.format('csv').option("header", "true").load("s3://supply-chain-management-project/delivery_events.csv")
delivery_Event_df.createOrReplaceTempView("delivery_Event")
display(delivery_Event_df)

In [0]:
driver_monthly_metrics_df= spark.read.format('csv').option("header", "true").load("s3://supply-chain-management-project/driver_monthly_metrics.csv")
driver_monthly_metrics_df.createOrReplaceTempView("driver_monthly_metrics")
display(driver_monthly_metrics_df)

In [0]:
drivers_df= spark.read.format('csv').option("header", "true").load("s3://supply-chain-management-project/drivers.csv")
drivers_df.createOrReplaceTempView("drivers")
display(drivers_df)

## Use catalog

In [0]:
%sql
USE CATALOG supply_chain;
USE bronze;

## Create synthetic table - Inventory table

In [0]:
from pyspark.sql.functions import expr, rand, when,col

# Step 1: Base data
inventory_df = spark.range(100000).withColumn(
    "sku_id", expr("concat('SKU', lpad(id % 100000,6,'0'))")
).withColumn(
    "location_id", expr("concat('LOC', lpad(id % 50,3,'0'))")
).withColumn(
    "on_hand_quantity", (rand() * 500).cast("int")
).withColumn(
    "intransit_qty", (rand() * 200).cast("int")
)

# Step 2: Introduce NULL values (~10%)
inventory_df = inventory_df.withColumn(
    "on_hand_quantity",
    when(rand() < 0.1, None).otherwise(col("on_hand_quantity"))
).withColumn(
    "intransit_qty",
    when(rand() < 0.1, None).otherwise(col("intransit_qty"))
)

# Step 3: Create duplicates (10% extra rows)
duplicate_df = inventory_df.sample(fraction=0.1)

final_inventory_df = inventory_df.union(duplicate_df).drop("id")

# Step 4: Save table
final_inventory_df.write.format("delta").mode("overwrite") \
    .saveAsTable("supply_chain.bronze.inventory")

## create synthetic table - purchase_orders table

In [0]:
from pyspark.sql.functions import expr, rand, current_date, date_sub, when, col

# Step 1: Base PO data
po_df = spark.range(100000).withColumn(
    "p_id", expr("concat('PO', lpad(id,6,'0'))")
).withColumn(
    "sup_id", expr("concat('SUP', lpad(id % 5000,5,'0'))")
).withColumn(
    "ordered_qty", (rand() * 500).cast("int")
).withColumn(
    "received_qty", (rand() * 500).cast("int")
).withColumn(
    "order_date", date_sub(current_date(), (rand() * 365).cast("int"))
)

# Step 2: Introduce NULL values (~5-10%)
po_df = po_df.withColumn(
    "received_qty",
    when(rand() < 0.1, None).otherwise(col("received_qty"))
).withColumn(
    "order_date",
    when(rand() < 0.05, None).otherwise(col("order_date"))
)

# Step 3: Create duplicates (~10%)
duplicate_df = po_df.sample(fraction=0.1)

final_po_df = po_df.union(duplicate_df).drop("id")

# Step 4: Save to Bronze
final_po_df.write.format("delta").mode("overwrite") \
    .saveAsTable("supply_chain.bronze.purchase_orders")

## View schema

In [0]:
spark.table("supply_chain.bronze.purchase_orders").printSchema()

## creating synthetic table for sales table

In [0]:
from pyspark.sql.functions import expr, rand, current_date, date_sub, col, when

# Step 1: Generate base data
sales_df = spark.range(100000).withColumn(
    "sales_id", expr("concat('S', lpad(id,6,'0'))")
).withColumn(
    "order_id", expr("concat('O', lpad(id,6,'0'))")
).withColumn(
    "customer_id", expr("concat('C', lpad(id % 20000,5,'0'))")
).withColumn(
    "sku_id", expr("concat('SKU', lpad(id % 100000,6,'0'))")
).withColumn(
    "location_id", expr("concat('LOC', lpad(id % 50,3,'0'))")
).withColumn(
    "quantity_sold", (rand() * 10 + 1).cast("int")
).withColumn(
    "unit_price", (rand() * 1000).cast("double")
).withColumn(
    "order_date", date_sub(current_date(), (rand() * 365).cast("int"))
).withColumn(
    "load_id", expr("concat('L', lpad(id % 50000,6,'0'))")
)

# Step 2: Calculate total amount
sales_df = sales_df.withColumn(
    "total_amount", col("quantity_sold") * col("unit_price")
)

# Step 3: Introduce NULLs (~5%)
sales_df = sales_df.withColumn(
    "unit_price",
    when(rand() < 0.05, None).otherwise(col("unit_price"))
)

# Step 4: Add duplicates (~10%)
duplicate_df = sales_df.sample(fraction=0.1)

final_sales_df = sales_df.union(duplicate_df).drop("id")

# Step 5: Save to Bronze
final_sales_df.write.format("delta").mode("overwrite") \
    .saveAsTable("supply_chain.bronze.sales")

In [0]:
customers_df= spark.read.format('csv').option("header", "true").load("s3://supply-chain-management-project/customers.csv")
customers_df.createOrReplaceTempView("customers")
display(customers_df)

## Ingest Kaggle dataset to Bronze

In [0]:
from pyspark.sql.functions import current_timestamp

loads_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("s3://supply-chain-management-project/loads.csv")

# Add metadata
loads_df = loads_df.withColumn("_ingest_ts", current_timestamp())

# Save to Bronze
loads_df.write.format("delta").mode("overwrite") \
    .saveAsTable("supply_chain.bronze.loads")

In [0]:
from pyspark.sql.functions import current_timestamp

# List of CSV tables to write to Bronze
csv_tables = [
    "customers", "delivery_events", "driver_monthly_metrics",
    "drivers", "facilities", "fuel_purchases"
]

for table_name in csv_tables:
    file_path = f"s3://supply-chain-management-project/{table_name}.csv"
    
    # Read CSV
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(file_path)
    
    # Add ingest timestamp
    df = df.withColumn("_ingest_ts", current_timestamp())
    
    # Write to Bronze
    df.write.format("delta").mode("overwrite") \
        .saveAsTable(f"supply_chain.bronze.{table_name}")
    
    print(f"✓ Written {table_name} to supply_chain.bronze.{table_name}")

print("\n✅ All CSV tables written to Bronze layer!")

In [0]:
from pyspark.sql.functions import current_timestamp

# List of remaining tables to write to Bronze
remaining_tables = [
    "maintenance_records", "routes", "safety_incidents",
    "trailers", "trips", "truck_utilization_metrics", "trucks"
]

for table_name in remaining_tables:
    file_path = f"s3://supply-chain-management-project/{table_name}.csv"
    
    # Read CSV
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(file_path)
    
    # Add ingest timestamp
    df = df.withColumn("_ingest_ts", current_timestamp())
    
    # Write to Bronze
    df.write.format("delta").mode("overwrite") \
        .saveAsTable(f"supply_chain.bronze.{table_name}")
    
    print(f"✓ Written {table_name} to supply_chain.bronze.{table_name}")

print("\n✅ All 7 remaining tables ingested to Bronze layer!")

## Ingest synthetic tables

In [0]:
final_sales_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("supply_chain.bronze.sales")
final_po_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("supply_chain.bronze.purchase_orders")
final_inventory_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("supply_chain.bronze.inventory")

In [0]:
# Load all bronze tables from supply_chain.bronze catalog
customers_df = spark.table("supply_chain.bronze.customers")
drivers_df = spark.table("supply_chain.bronze.drivers")
driver_monthly_metrics_df = spark.table("supply_chain.bronze.driver_monthly_metrics")
delivery_events_df = spark.table("supply_chain.bronze.delivery_events")
facilities_df = spark.table("supply_chain.bronze.facilities")
fuel_purchases_df = spark.table("supply_chain.bronze.fuel_purchases")
maintenance_records_df = spark.table("supply_chain.bronze.maintenance_records")
routes_df = spark.table("supply_chain.bronze.routes")
safety_incidents_df = spark.table("supply_chain.bronze.safety_incidents")
trailers_df = spark.table("supply_chain.bronze.trailers")
trips_df = spark.table("supply_chain.bronze.trips")
truck_utilization_metrics_df = spark.table("supply_chain.bronze.truck_utilization_metrics")
trucks_df = spark.table("supply_chain.bronze.trucks")
loads_df = spark.table("supply_chain.bronze.loads")
sales_df = spark.table("supply_chain.bronze.sales")
purchase_orders_df = spark.table("supply_chain.bronze.purchase_orders")
inventory_df = spark.table("supply_chain.bronze.inventory")

print("✅ All 17 bronze tables loaded from supply_chain.bronze!")

## Validate Bronze table

In [0]:
%sql
SELECT COUNT(*) FROM supply_chain.bronze.loads;

In [0]:
spark.sql("DESCRIBE DETAIL supply_chain.bronze.inventory").show(truncate=False)

In [0]:
spark.table("supply_chain.bronze.inventory").printSchema()